# MarketPulse: 02 - Demand Regimes & Price Elasticity Analysis

This research notebook analyzes demand patterns using the **Syntetos-Boylan (2005)** categorization framework and investigates empirical price elasticity across product categories.

### Framework Formulation:
- **Average Demand Interval (ADI)**: Average spacing between periods with non-zero demand:
  $$ADI = \frac{N}{\sum_{t=1}^N \mathbb{I}(y_t > 0)}$$
  *Cutoff*: $ADI = 1.32$

- **Squared Coefficient of Variation ($CV^2$)**: Variance of non-zero transaction sizes:
  $$CV^2 = \left( \frac{\sigma_{nz}}{\mu_{nz}} \right)^2$$
  *Cutoff*: $CV^2 = 0.49$

### 4 Distinct Regimes:
1. **Smooth**: $ADI < 1.32$ and $CV^2 < 0.49$ (Regular cadence, low size variance)
2. **Intermittent**: $ADI \ge 1.32$ and $CV^2 < 0.49$ (Sporadic cadence, consistent order size)
3. **Erratic**: $ADI < 1.32$ and $CV^2 \ge 0.49$ (Regular cadence, highly volatile order sizes)
4. **Lumpy**: $ADI \ge 1.32$ and $CV^2 \ge 0.49$ (Sporadic cadence and highly volatile sizes)

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import load_config
from src.features.demand_features import classify_demand_regimes
from src.features.price_features import calculate_price_features, estimate_price_elasticity

sns.set_theme(style="whitegrid", palette="muted")
config = load_config(project_root / "configs" / "config.yaml")
df = pd.read_parquet(config.get_processed_path(config.data.processed_file))
print(f"Loaded {len(df):,} records.")

## 1. Syntetos-Boylan Regime Classification

In [ ]:
regimes_df = classify_demand_regimes(df)
print("Regime Breakdown:")
display(regimes_df["regime"].value_counts(normalize=True).mul(100).round(1).to_frame("Percentage (%)"))

# Syntetos-Boylan Quadrant Scatter Plot
plt.figure(figsize=(10, 6))
palette = {"Smooth": "#2ecc71", "Intermittent": "#f1c40f", "Erratic": "#e67e22", "Lumpy": "#e74c3c"}

for reg, color in palette.items():
    sub = regimes_df[regimes_df["regime"] == reg]
    if len(sub) > 0:
        plt.scatter(sub["adi"], sub["cv2"], label=f"{reg} (n={len(sub)})", color=color, alpha=0.8, s=60)

plt.axvline(1.32, color="black", linestyle="--", label="ADI Cutoff (1.32)")
plt.axhline(0.49, color="black", linestyle=":", label="CV² Cutoff (0.49)")
plt.title("Syntetos-Boylan Demand Categorization Matrix")
plt.xlabel("Average Demand Interval (ADI)")
plt.ylabel("Squared Coefficient of Variation (CV²)")
plt.legend()
plt.show()

## 2. Autocorrelation & Memory Across Regimes

In [ ]:
autocorr_summary = regimes_df.groupby("regime", observed=True)[["autocorr_lag1", "autocorr_lag7", "autocorr_lag28"]].mean()
print("Average Autocorrelation by Demand Regime:")
display(autocorr_summary.round(3))

autocorr_summary.plot(kind="bar", figsize=(10, 4))
plt.title("Autocorrelation Structure Across Demand Regimes")
plt.ylabel("Pearson Correlation")
plt.xticks(rotation=0)
plt.show()

## 3. Empirical Price Elasticity of Demand

We estimate the log-log specification: $\ln(Q + 1) = \alpha + \beta \ln(P) + \epsilon$

In [ ]:
df_price = calculate_price_features(df)
elasticity_df = estimate_price_elasticity(df_price, group_col="cat_id")
print("Price Elasticity Estimates:")
display(elasticity_df)

plt.figure(figsize=(8, 4))
colors = ["#2ecc71" if e < -1.0 else "#e67e22" for e in elasticity_df["elasticity"]]
plt.barh(elasticity_df["group"], elasticity_df["elasticity"], color=colors)
plt.axvline(-1.0, color="red", linestyle="--", label="Unit Elastic (-1.0)")
plt.title("Estimated Price Elasticity by Category")
plt.xlabel("Elasticity Beta (log Q / log P)")
plt.legend()
plt.show()